# UCInsure: Flood Model

An end-to-end actuarial ML pipeline combining flood-risk data ingestion, exploratory analysis, machine learning, and visualization using **FEMA NFIP** claims data.

## Pipeline Stages

| # | Stage | Description |
|---|-------|-------------|
| 1 | **Setup & Imports** | Install dependencies and configure global constants |
| 2 | **Data Loading** | Fetch and cache FEMA OpenFEMA NFIP claims via REST API |
| 3 | **Exploratory Data Analysis** | Understand dataset shape, distributions, and missing values |
| 4 | **Feature Engineering** | Derive temporal features, encode types, create 3-class risk target |

## Data Source
- FEMA OpenFEMA API v2 — `FimaNfipClaims`
- Cached at `data/cache/FimaNfipClaimsV2.csv` after first run
- Sample size: 250,000 rows (configurable via `SAMPLE_ROWS`)

## 1. Setup & Imports

Install required packages and configure global constants shared across all pipeline stages.

In [ ]:
%pip install --break-system-packages numpy pandas scikit-learn joblib matplotlib seaborn

import json
import math
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import joblib
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# ── Global constants ──────────────────────────────────────────────────────────
DATA_URL        = "https://www.fema.gov/api/open/v2/FimaNfipClaims"
DATA_CACHE_PATH = Path("data") / "cache" / "FimaNfipClaimsV2.csv"
MODEL_CACHE_DIR = Path("data") / "cache" / "models"

ALLOWED_COLUMNS = (
    "reportedCity",
    "reportedZipCode",
    "latitude",
    "longitude",
    "floodEvent",
    "dateOfLoss",
    "yearOfLoss",
    "floodZoneCurrent",
    "waterDepth",
    "numberOfFloorsInTheInsuredBuilding",
    "occupancyType",
    "primaryResidenceIndicator",
    "buildingPropertyValue",
    "contentsPropertyValue",
    "amountPaidOnBuildingClaim",
    "amountPaidOnContentsClaim",
    "buildingDamageAmount",
)

SAMPLE_ROWS  = 250_000
RANDOM_STATE = 42

print("✓ Imports and constants ready.")


## 2. Data Loading

Fetch FEMA OpenFEMA NFIP claims via REST API with sequential pagination. Results are cached locally so subsequent runs load instantly from disk.

In [ ]:
def load_claims_data(
    data_url: str = DATA_URL,
    max_rows: int | None = SAMPLE_ROWS,
    cache_path: Path = DATA_CACHE_PATH,
) -> pd.DataFrame:
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    if cache_path.exists():
        print(f"Cache found — loading from {cache_path}")
        df = pd.read_csv(
            cache_path,
            usecols=list(ALLOWED_COLUMNS),
            low_memory=False,
            nrows=max_rows,
        )
        return df

    page_size   = 10_000 if max_rows is None else min(10_000, max_rows)
    target_rows = float("inf") if max_rows is None else max_rows
    frames: list[pd.DataFrame] = []
    fetched_rows = 0
    skip = 0

    while fetched_rows < target_rows:
        query = urlencode(
            {
                "$select": ",".join(ALLOWED_COLUMNS),
                "$top":    page_size,
                "$skip":   skip,
            }
        )
        request = Request(
            f"{data_url}?{query}",
            headers={
                "User-Agent": "Mozilla/5.0 (compatible; UCInsure/1.0)",
                "Accept":     "application/json",
            },
        )
        with urlopen(request) as response:
            payload = json.loads(response.read().decode("utf-8"))

        batch = pd.DataFrame.from_records(
            payload["FimaNfipClaims"], columns=list(ALLOWED_COLUMNS)
        )
        if batch.empty:
            break

        frames.append(batch)
        fetched_rows += len(batch)
        skip += len(batch)

        if len(batch) < page_size:
            break

    if not frames:
        raise ValueError("No FEMA NFIP claims rows were returned from the OpenFEMA API.")

    df = pd.concat(frames, ignore_index=True)
    if max_rows is not None:
        df = df.head(max_rows).copy()

    df.to_csv(cache_path, index=False)
    print(f"Fetched {len(df):,} rows → cached at {cache_path}")
    return df


raw_df = load_claims_data()
print(f"\nDataset shape: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns")

## 3. Exploratory Data Analysis

Inspect dataset shape, column types, and missing-value distribution before any transformations.

In [ ]:
print(f"Shape : {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns")
print(f"\nColumn dtypes:")
print(raw_df.dtypes.to_string())

missing = raw_df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
if not missing.empty:
    print(f"\nMissing values (columns with any NaN):")
    print(missing.to_string())
else:
    print("\nNo missing values detected.")

print(f"\nNumeric summary:")
raw_df.describe()

## 4. Feature Engineering

Derive temporal features (`lossMonth`, `lossDayOfYear`, `totalClaimAmount`), enforce numeric / string types, and create the 3-class `riskLevel` target from `buildingDamageAmount` percentile thresholds.

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    frame = df.copy()

    frame["dateOfLoss"]    = pd.to_datetime(frame["dateOfLoss"], errors="coerce")
    frame["lossMonth"]     = frame["dateOfLoss"].dt.month
    frame["lossDayOfYear"] = frame["dateOfLoss"].dt.dayofyear

    numeric_cols = [
        "latitude", "longitude", "yearOfLoss", "waterDepth",
        "numberOfFloorsInTheInsuredBuilding", "buildingPropertyValue",
        "contentsPropertyValue", "amountPaidOnBuildingClaim",
        "amountPaidOnContentsClaim", "buildingDamageAmount",
    ]
    categorical_cols = [
        "reportedCity", "reportedZipCode", "floodEvent",
        "floodZoneCurrent", "occupancyType", "primaryResidenceIndicator",
    ]

    for col in numeric_cols:
        frame[col] = pd.to_numeric(frame[col], errors="coerce")
    for col in categorical_cols:
        frame[col] = frame[col].astype("string").fillna("missing")

    frame["totalClaimAmount"] = (
        frame["amountPaidOnBuildingClaim"].fillna(0)
        + frame["amountPaidOnContentsClaim"].fillna(0)
    )
    return frame


def create_risk_target(df: pd.DataFrame) -> pd.Series:
    """Bin buildingDamageAmount into low / medium / high using 33rd and 66th percentile thresholds."""
    damage = df["buildingDamageAmount"]
    low_t  = damage.quantile(0.33)
    high_t = damage.quantile(0.66)
    risk = pd.cut(
        damage,
        bins=[-float("inf"), low_t, high_t, float("inf")],
        labels=["low", "medium", "high"],
        include_lowest=True,
    )
    return risk.astype("string")


# ── Column lists used by all downstream cells ─────────────────────────────────
feature_columns = [
    "reportedCity", "reportedZipCode", "latitude", "longitude",
    "floodEvent", "yearOfLoss", "floodZoneCurrent", "waterDepth",
    "numberOfFloorsInTheInsuredBuilding", "occupancyType",
    "primaryResidenceIndicator", "buildingPropertyValue",
    "contentsPropertyValue", "amountPaidOnBuildingClaim",
    "amountPaidOnContentsClaim", "lossMonth", "lossDayOfYear",
    "totalClaimAmount",
]
numeric_features = [
    "latitude", "longitude", "yearOfLoss", "waterDepth",
    "numberOfFloorsInTheInsuredBuilding", "buildingPropertyValue",
    "contentsPropertyValue", "amountPaidOnBuildingClaim",
    "amountPaidOnContentsClaim", "lossMonth", "lossDayOfYear",
    "totalClaimAmount",
]
categorical_features = [
    "reportedCity", "reportedZipCode", "floodEvent",
    "floodZoneCurrent", "occupancyType", "primaryResidenceIndicator",
]

df = engineer_features(raw_df)
df["riskLevel"] = create_risk_target(df)
df = df.dropna(subset=["riskLevel"]).copy()

print(f"Engineered dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nRisk level distribution:\n{df['riskLevel'].value_counts().sort_index().to_string()}")